# Solow Growth Model Simulation (Dynare + Octave)

This notebook demonstrates how to implement and simulate the Solow growth model using Dynare inside Google Colab.

We combine:
- computational tools (Dynare + Octave)
- economic theory (Solow growth model)

The goal is to understand how economies evolve over time and converge to a steady state.

In [ ]:
!apt update
!apt install -y octave dynare

## Installing Dynare and GNU Octave

This step prepares the computational environment.

- `apt update`: refreshes the system package list
- `apt install`: installs required tools

### Tools used
- **GNU Octave**: numerical computing environment
- **Dynare**: macroeconomic modeling toolkit

These tools allow us to solve and simulate dynamic economic models such as DSGE and growth models.

In [ ]:
%%bash

cat <<EOF > model1.mod
var y;
model;
y = 0.9*y;
end;
EOF

octave --eval "addpath('/usr/lib/dynare/matlab'); dynare model1.mod"

## Minimal Dynare Model (Sanity Check)

We begin with a simple model to verify the setup.

### Equation
$$ y_t = 0.9 y_t $$

This is not economically meaningful.

### Purpose
- verify Dynare installation
- ensure `.mod` files execute correctly

Once this works, we proceed to the full economic model.

## Solow Growth Model

We now implement the Solow growth model, one of the most important models in macroeconomics.

### Production Function
$$ y_t = A k_{t-1}^{\alpha} $$

- Output depends on capital
- Technology determines productivity

---

### Capital Accumulation
$$ k_t = \frac{s y_t + (1-\delta)k_{t-1}}{(1+n)(1+g)} $$

- Savings increases capital
- Depreciation reduces capital
- Population and technology dilute capital

---

### Key Economic Insight

The model predicts convergence to a steady state driven by diminishing returns to capital.

In [ ]:
%%bash

cat <<EOF > solow.mod
var y k c w r sav;
varexo a x z;

parameters alpha delta n g s;

alpha = 0.333;
delta = 0.03;
n = 0.01;
g = 0.02;
s = 0.30;

model;
y = a*(k(-1)^alpha);
c = (1-(s*x))*y;
k = (1/((1+n*z)*(1+g)))*((s*x*y) + (1-delta)*k(-1));
r = alpha*a*k(-1)^(alpha-1) - delta;
w = (1-alpha)*a*k(-1)^alpha;
sav = s*x;
end;

initval;
k = 5.3;
c = 1.38;
y = 1.7;
a = 1;
r = 0.075;
w = 1.16;
x = 1;
z = 1;
end;

steady;

endval;
z = 1.01;
end;

steady;
check;

perfect_foresight_setup(periods=100);
perfect_foresight_solver;

save('solow_results.mat', 'oo_');
EOF

octave --eval "addpath('/usr/lib/dynare/matlab'); dynare solow.mod"

## Transition Dynamics

We simulate a change in the economy using:
`endval`

Dynare computes the full transition path using perfect foresight.

### Economic Interpretation
- The economy starts at a steady state
- A shock shifts the system
- The economy gradually adjusts to a new steady state

This reflects real-world capital accumulation dynamics.

In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt

data = sio.loadmat('solow_results.mat')
endo = data['oo_']['endo_simul'][0][0]

names = ['y','k','c','w','r','sav']

for i, name in enumerate(names):
    plt.figure()
    plt.plot(endo[i])
    plt.title(name)
    plt.xlabel('Time')
    plt.show()

## Interpreting Results

- Capital increases over time
- Output follows capital
- Consumption depends on saving behavior
- Interest rates decline as capital accumulates

### Core Result

The Solow model predicts:

> Saving affects output levels, not long-run growth rates.
